# DAY4 과제 01 : 기초문제

데이터 파일 : `packaging_line.csv` (포장 라인 A · B · C 에서 배치 12개씩, 모두 36개 배치의 처리시간 기록)

- 유의수준은 모든 문제에서 α = 0.05 로 한다
- 결론은 두 줄로 쓴다 — 통계적 결론(귀무가설을 기각한다 / 기각하지 못한다)과 사실적 결론(현장 말로 풀어 쓴 문장)
- 기각하지 못하면 "같다"가 아니라 "다르다고 볼 근거가 부족하다"로 쓴다

| 컬럼 | 설명 |
|---|---|
| batch_id | 배치 번호 (A01 ~ C12) |
| line | 포장 라인 (A / B / C) |
| processing_time_sec | 배치 처리시간 (초) |

In [ ]:
%pip install -q koreanize-matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from scipy import stats

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"
csv_path = os.path.join(data_folder, "packaging_line.csv")

df = pd.read_csv(csv_path)

data = df.copy()

## 1부. 일원분산분석 : 포장 라인 세 곳의 처리시간 비교

### 1. 라인별 배치 수 · 평균 · 표준편차를 한 표로 구하시오. (소수 둘째 자리까지)

In [ ]:
line_summary = data.groupby("line")["processing_time_sec"].agg(["count", "mean", "std"]).round(2)
#   groupby("line") : 라인별로 묶는다 / agg([...]) : 개수 · 평균 · 표준편차를 한 번에 구한다
#   std 는 n − 1 로 나누는 표본표준편차가 기본이다
line_summary

,count,mean,std
line,,,
A,12,50.0,2.2
B,12,51.0,2.4
C,12,53.2,2.6


### 2. 라인별 처리시간을 `A`, `B`, `C` 세 변수로 꺼내고, 라인마다 Shapiro-Wilk 정규성 검정의 p값을 구하시오.

- 귀무가설(H₀): 이 라인의 처리시간은 정규분포를 따른다.
- 대립가설(H₁): 이 라인의 처리시간은 정규분포를 따르지 않는다.

In [ ]:
A = data.loc[data["line"] == "A", "processing_time_sec"]   # 조건(line 이 A)에 맞는 행의 처리시간 열만 꺼낸다
B = data.loc[data["line"] == "B", "processing_time_sec"]
C = data.loc[data["line"] == "C", "processing_time_sec"]

print("Shapiro p A:", round(stats.shapiro(A).pvalue, 3))     # 라인마다 따로 검정한다
print("Shapiro p B:", round(stats.shapiro(B).pvalue, 3))
print("Shapiro p C:", round(stats.shapiro(C).pvalue, 3))

Shapiro p A: 0.443
Shapiro p B: 0.565
Shapiro p C: 0.277


### 3. Levene 검정으로 세 라인의 분산이 같은지 확인하시오.

- 귀무가설(H₀): 세 라인 처리시간의 모분산은 같다. (σA² = σB² = σC²)
- 대립가설(H₁): 적어도 한 라인의 모분산이 다르다.

In [ ]:
print("Levene p:", round(stats.levene(A, B, C).pvalue, 3))   # 세 집단을 쉼표로 이어 넣는다

Levene p: 0.73


### 4. 일원분산분석으로 세 라인의 평균 처리시간이 같은지 검정하고, F 와 p값을 구하시오.

- 귀무가설(H₀): 세 라인의 모평균 처리시간은 같다. (μA = μB = μC)
- 대립가설(H₁): 적어도 한 라인의 모평균 처리시간이 다르다. (적어도 한 μ가 다르다)

In [ ]:
anova = stats.f_oneway(A, B, C)          # 세 라인을 한 번에 비교한다
print("F:", round(anova.statistic, 2))
print("p값:", round(anova.pvalue, 4))

F: 5.56
p값: 0.0083


### 5. 4번에서 귀무가설을 기각했다면, Tukey HSD 사후검정으로 어느 라인끼리 다른지 확인하시오.

아래 두 방법 중 **하나를 골라** 쓴다.
- 방법 1 : `pairwise_tukeyhsd` (statsmodels)
- 방법 2 : `stats.tukey_hsd` (scipy)

쌍마다
- 귀무가설(H₀): 두 라인의 모평균 처리시간은 같다. (예 : μA = μC)
- 대립가설(H₁): 두 라인의 모평균 처리시간은 다르다. (예 : μA ≠ μC)

In [ ]:
# 방법 1 : pairwise_tukeyhsd
from statsmodels.stats.multicomp import pairwise_tukeyhsd

print(pairwise_tukeyhsd(data["processing_time_sec"], data["line"]))
#   (값 열, 이름표 열) 순서로 넣는다. 한 줄이 한 쌍이다
#   meandiff = group2 평균 − group1 평균 / p-adj = 조정 p값 / reject True = 차이가 있다

# 방법 2 : stats.tukey_hsd
tukey = stats.tukey_hsd(A, B, C)          # 넣은 순서대로 0 = A, 1 = B, 2 = C
print(tukey)                              # Statistic = 앞 집단 평균 − 뒤 집단 평균
print("A − B 조정 p값:", round(tukey.pvalue[0, 1], 3))   # pvalue[0, 1] : 0번(A)과 1번(B) 쌍
print("A − C 조정 p값:", round(tukey.pvalue[0, 2], 3))
print("B − C 조정 p값:", round(tukey.pvalue[1, 2], 3))

Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
     A      B      1.0 0.5706 -1.4092 3.4092  False
     A      C   3.2008 0.0071  0.7917   5.61   True
     B      C   2.2008 0.0789 -0.2083   4.61  False
---------------------------------------------------
Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -1.000     0.571    -3.409     1.409
 (0 - 2)     -3.201     0.007    -5.610    -0.792
 (1 - 0)      1.000     0.571    -1.409     3.409
 (1 - 2)     -2.201     0.079    -4.610     0.208
 (2 - 0)      3.201     0.007     0.792     5.610
 (2 - 1)      2.201     0.079    -0.208     4.610

A − B 조정 p값: 0.571
A − C 조정 p값: 0.007
B − C 조정 p값: 0.079


## 2부. 비율 검정 : 한 집단 · 두 집단

### 6. 출하 제품 120개를 검사했더니 불량이 13개였다. 기존 불량률 기준은 5%다. 표본 불량률, 정확 이항검정의 p값, 모비율의 95% 신뢰구간을 구하시오.

- 귀무가설(H₀): 출하 제품의 모집단 불량률은 5%다. (p = 0.05)
- 대립가설(H₁): 출하 제품의 모집단 불량률은 5%가 아니다. (p ≠ 0.05)

In [ ]:
k = 13          # 불량 개수 (정수로 넣는다)
n = 120         # 검사한 전체 개수
p0 = 0.05       # 기준 불량률 5%

print("표본 불량률:", round(k / n * 100, 1), "%")

ship_result = stats.binomtest(k=k, n=n, p=p0)                  # alternative 를 적지 않으면 양측 검정
ship_ci = ship_result.proportion_ci(confidence_level=0.95)    # 모집단 불량률의 95% 신뢰구간
print("p값:", round(ship_result.pvalue, 4))
print("모비율 95% 신뢰구간:", round(ship_ci.low * 100, 1), "~", round(ship_ci.high * 100, 1), "%")

표본 불량률: 10.8 %
p값: 0.0093
모비율 95% 신뢰구간: 5.9 ~ 17.8 %


### 7. 주간조와 야간조에서 제품을 100개씩 검사했다. 주간조는 불량 17개, 야간조는 불량 9개였다.

(1) 아래 교차표를 만들고, 기대빈도가 모두 5 이상인지 확인하시오.

| 근무조 | 불량 | 정상 |
|---|---|---|
| 주간 | 17 | 83 |
| 야간 | 9 | 91 |

(2) 카이제곱 검정의 χ² 와 p값을 구하시오.

(3) 불량률 차이(주간 − 야간)와 그 95% 신뢰구간을 구하시오. (`method="wald"`)

- 귀무가설(H₀): 주간조와 야간조의 모집단 불량률은 같다. (p_주간 = p_야간)
- 대립가설(H₁): 주간조와 야간조의 모집단 불량률은 다르다. (p_주간 ≠ p_야간)

In [ ]:
# (1) 교차표와 기대빈도
shift_table = pd.DataFrame({"불량": [17, 9], "정상": [83, 91]}, index=["주간", "야간"])
print(shift_table)
print(pd.DataFrame(stats.contingency.expected_freq(shift_table), index=shift_table.index, columns=shift_table.columns))
#   기대빈도 = 행 합계 × 열 합계 ÷ 전체 : 두 근무조의 불량률이 같다면 나왔을 건수

# (2) 카이제곱 검정 : 건수 교차표를 그대로 넣는다
shift_result = stats.chi2_contingency(shift_table)
print("χ²:", round(shift_result.statistic, 2), "/ p값:", round(shift_result.pvalue, 4))

# (3) 불량률 차이와 95% 신뢰구간
from statsmodels.stats.proportion import confint_proportions_2indep

diff_low, diff_high = confint_proportions_2indep(17, 100, 9, 100, method="wald")
#   (주간 불량 수, 주간 전체 수, 야간 불량 수, 야간 전체 수) 순서 → 차이 = 주간 − 야간
print("불량률 주간:", 17 / 100 * 100, "% / 야간:", 9 / 100 * 100, "%")
print("불량률 차이(주간 − 야간):", round((17 / 100 - 9 / 100) * 100, 1), "%p")
print("차이 95% 신뢰구간:", round(diff_low * 100, 2), "~", round(diff_high * 100, 2), "%p")

    불량  정상
주간  17  83
야간   9  91
      불량    정상
주간  13.0  87.0
야간  13.0  87.0
χ²: 2.17 / p값: 0.1411
불량률 주간: 17.0 % / 야간: 9.0 %
불량률 차이(주간 − 야간): 8.0 %p
차이 95% 신뢰구간: -1.26 ~ 17.26 %p


## 3부. 카이제곱 검정 : 적합도 · 동질성 · 독립성

### 8. 라벨 오류 100건을 유형별로 세었더니 정보 누락 42건, 위치 오류 38건, 인쇄 흐림 20건이었다. 과거 비율은 50% · 30% · 20%다. 기대빈도를 구하고 적합도 검정의 χ² 와 p값을 구하시오.

- 귀무가설(H₀): 라벨 오류 유형의 모비율은 과거 비율과 같다. (p₁ = 0.50, p₂ = 0.30, p₃ = 0.20)
- 대립가설(H₁): 적어도 한 오류 유형의 모비율이 과거 비율과 다르다.

In [ ]:
f_obs = np.array([42, 38, 20])          # 관측빈도 : 실제로 센 건수
ratio = np.array([0.50, 0.30, 0.20])     # 과거 비율 (합계 1)
f_exp = f_obs.sum() * ratio              # 기대빈도 = 100 × 과거 비율 → 50 · 30 · 20
print("기대빈도:", f_exp)

label_result = stats.chisquare(f_obs=f_obs, f_exp=f_exp)   # 관측과 기대의 합계가 같아야 한다
print("χ²:", round(label_result.statistic, 2), "/ p값:", round(label_result.pvalue, 4))

기대빈도: [50. 30. 20.]
χ²: 3.41 / p값: 0.1815


### 9. 교대조 1조 · 2조 · 3조에서 제품을 200개씩 정해서 뽑아 검사했다. 불량은 각각 10개, 12개, 24개였다. 교차표를 만들고 기대빈도를 확인한 뒤, 동질성 검정의 χ² 와 p값을 구하시오.

- 귀무가설(H₀): 세 교대조의 모집단 불량률은 같다. (p₁ = p₂ = p₃)
- 대립가설(H₁): 적어도 한 교대조의 모집단 불량률이 다르다.

In [ ]:
team_table = pd.DataFrame({"불량": [10, 12, 24], "정상": [190, 188, 176]}, index=["1조", "2조", "3조"])
#   정상 = 200 − 불량 → 190 · 188 · 176
print(team_table)
print("불량률(%):", (team_table["불량"] / 200 * 100).tolist())
print(pd.DataFrame(stats.contingency.expected_freq(team_table), index=team_table.index, columns=team_table.columns).round(1))

team_result = stats.chi2_contingency(team_table)
print("χ²:", round(team_result.statistic, 2), "/ p값:", round(team_result.pvalue, 4))

    불량   정상
1조  10  190
2조  12  188
3조  24  176
불량률(%): [5.0, 6.0, 12.0]
      불량     정상
1조  15.3  184.7
2조  15.3  184.7
3조  15.3  184.7
χ²: 8.1 / p값: 0.0174


### 10. 한 달 동안 나온 불량 200건을 설비와 불량 유형 두 가지로 분류했다. 비율표(설비별 100%)를 구하고, 독립성 검정의 χ² 와 p값을 구하시오.

| 설비 | 스크래치 | 치수 | 이물 |
|---|---|---|---|
| 1호기 | 38 | 27 | 15 |
| 2호기 | 42 | 33 | 45 |

- 귀무가설(H₀): 설비와 불량 유형은 독립이다. 설비가 달라도 불량 유형 비율은 같다. (P(설비 ∩ 유형) = P(설비) × P(유형))
- 대립가설(H₁): 설비와 불량 유형은 독립이 아니다. 설비에 따라 불량 유형 비율이 다르다. (P(설비 ∩ 유형) ≠ P(설비) × P(유형))

In [ ]:
machine_table = pd.DataFrame({"스크래치": [38, 42], "치수": [27, 33], "이물": [15, 45]}, index=["1호기", "2호기"])
print(machine_table)

machine_ratio = machine_table.div(machine_table.sum(axis=1), axis=0) * 100
#   sum(axis=1) : 행(설비)마다 합계 → 80 · 120 / div(..., axis=0) : 행마다 그 합계로 나눈다
print(machine_ratio.round(1))
print(pd.DataFrame(stats.contingency.expected_freq(machine_table), index=machine_table.index, columns=machine_table.columns).round(1))

machine_result = stats.chi2_contingency(machine_table)   # 검정에는 비율표가 아니라 건수표를 넣는다
print("χ²:", round(machine_result.statistic, 2), "/ p값:", round(machine_result.pvalue, 4))

     스크래치  치수  이물
1호기    38  27  15
2호기    42  33  45
     스크래치    치수    이물
1호기  47.5  33.8  18.8
2호기  35.0  27.5  37.5
     스크래치    치수    이물
1호기  32.0  24.0  24.0
2호기  48.0  36.0  36.0
χ²: 8.12 / p값: 0.0172


## 확인문제

앞의 결과를 보고 답을 1~2문장으로 적으시오.

### 확인 1. 2번 · 3번 결과로 볼 때, 4번에서 일원분산분석을 써도 되는 이유는 무엇인가?

**답 :** 세 라인의 Shapiro p값(0.443 · 0.565 · 0.277)이 모두 0.05 이상이라 정규성이 크게 어긋나지 않고, Levene p값(0.730)도 0.05 이상이라 분산이 다르다고 볼 근거가 부족하다. 그래서 정규성과 등분산을 전제로 하는 일원분산분석을 쓴다.

### 확인 2. 4번의 통계적 결론과 사실적 결론을 쓰시오.

**답 :** 통계적 결론 : p = 0.0083 < 0.05 이므로 귀무가설을 기각한다.<br>사실적 결론 : 세 라인 중 적어도 한 라인의 평균 처리시간이 다르다고 볼 통계적 근거가 있다. 표본 평균은 A 50.00초, B 51.00초, C 53.20초다.

### 확인 3. 5번에서 차이가 있는 쌍은 어디이며, 어느 라인이 몇 초 더 긴가?

**답 :** A · C 쌍만 조정 p = 0.007 < 0.05 로 차이가 있다. 표본에서 C가 A보다 평균 3.20초 길었다. A · B(조정 p = 0.571)와 B · C(조정 p = 0.079)는 차이가 있다고 볼 근거가 부족하다.

### 확인 4. 6번의 통계적 결론과 사실적 결론을 쓰시오.

**답 :** 통계적 결론 : p = 0.0093 < 0.05 이므로 귀무가설을 기각한다.<br>사실적 결론 : 출하 제품의 불량률이 기준 5%와 다르다고 볼 통계적 근거가 있다. 표본 불량률은 10.8%로 기준보다 높고, 95% 신뢰구간은 5.9 ~ 17.8%다 (n = 120).

### 확인 5. 7번에서 표본 불량률은 주간이 8%p 높았다. 그런데도 결론이 어떻게 나왔는지 신뢰구간과 함께 쓰시오.

**답 :** 통계적 결론 : p = 0.1411 ≥ 0.05 이므로 귀무가설을 기각하지 못한다.<br>사실적 결론 : 주간조와 야간조의 불량률이 다르다고 볼 통계적 근거가 부족하다. 차이 8%p 의 95% 신뢰구간 −1.26 ~ 17.26%p 가 0 을 포함한다 (각 100개).

### 확인 6. 8번의 통계적 결론과 사실적 결론을 쓰시오.

**답 :** 통계적 결론 : p = 0.1815 ≥ 0.05 이므로 귀무가설을 기각하지 못한다.<br>사실적 결론 : 이번 라벨 오류 유형 비율이 과거 비율(50% · 30% · 20%)과 다르다고 볼 통계적 근거가 부족하다. 관측 42 · 38 · 20건, 기대 50 · 30 · 20건이다.

### 확인 7. 9번의 통계적 결론과 사실적 결론을 쓰시오.

**답 :** 통계적 결론 : p = 0.0174 < 0.05 이므로 귀무가설을 기각한다.<br>사실적 결론 : 세 교대조 중 적어도 한 조의 불량률이 다르다고 볼 통계적 근거가 있다. 표본 불량률은 1조 5%, 2조 6%, 3조 12%로 3조가 가장 높다 (각 200개).

### 확인 8. 10번의 통계적 결론과 사실적 결론을 쓰시오.

**답 :** 통계적 결론 : p = 0.0172 < 0.05 이므로 귀무가설을 기각한다.<br>사실적 결론 : 설비와 불량 유형은 독립이라고 보기 어렵다. 설비에 따라 불량 유형 비율이 다르다고 볼 통계적 근거가 있다. 이물 불량 비율이 1호기 18.8%, 2호기 37.5%로 차이가 가장 크다 (불량 200건).

## 개념 확인 (객관식)

알맞은 답의 번호를 하나 고르시오.

### 개념 1. 일원분산분석에서 귀무가설을 기각했다. 이 결과로 말할 수 있는 것은?

① 모든 집단의 평균이 서로 다르다  
② 적어도 한 집단의 평균이 다르다  
③ A와 C의 평균이 다르다  
④ 모든 집단의 평균이 같다

**답 :** ② — 일원분산분석은 '적어도 하나가 다르다'까지만 알려 준다. 어느 집단끼리 다른지는 사후검정으로 확인한다.

### 개념 2. 일원분산분석에서 기각한 뒤, 어느 집단끼리 다른지 확인하는 사후검정은?

① Shapiro-Wilk 검정  
② Levene 검정  
③ Tukey HSD  
④ 적합도 검정

**답 :** ③ — Tukey HSD 는 여러 쌍을 비교하면서 잘못 기각할 확률이 유의수준을 넘지 않게 보정한다.

### 개념 3. 세 집단 모두 정규성은 크게 어긋나지 않지만, Levene 검정에서 분산이 다르다고 나왔다. 평균 비교에 알맞은 방법은?

① 일원분산분석 `stats.f_oneway(a, b, c)`  
② Welch ANOVA `stats.f_oneway(a, b, c, equal_var=False)`  
③ Fisher 정확검정  
④ 적합도 검정

**답 :** ② — equal_var=False 는 분산이 같다고 가정하지 않는다는 뜻이다.

### 개념 4. 정규성이 크게 어긋난 세 집단을 비교할 때 쓰는 검정은?

① Kruskal-Wallis 검정  
② 2×2 카이제곱 검정  
③ 정확 이항검정  
④ Levene 검정

**답 :** ① — Kruskal-Wallis 는 값을 순위로 바꿔 비교하므로 정규분포를 가정하지 않는다.

### 개념 5. "제품 200개 중 불량 16개, 불량률이 기준 5%와 다른가"를 검정하는 함수는?

① `stats.binomtest`  
② `stats.f_oneway`  
③ `stats.levene`  
④ `stats.tukey_hsd`

**답 :** ① — 한 집단의 비율을 기준값과 비교하는 정확 이항검정이다. k(불량 수)와 n(전체 수)은 정수로 넣는다.

### 개념 6. 이번 달 불량 유형별 건수가 과거 비율과 맞는지 확인하는 검정은?

① 독립성 검정  
② 동질성 검정  
③ 적합도 검정  
④ 일원분산분석

**답 :** ③ — 범주형 변수 하나의 관측빈도를 기준 비율로 만든 기대빈도와 비교하는 적합도 검정이다 (`stats.chisquare`).

### 개념 7. 라인 A · B · C 에서 제품을 300개씩 정해서 뽑아, 세 라인의 불량률이 같은지 확인하는 검정은?

① 적합도 검정  
② 동질성 검정  
③ 정확 이항검정  
④ Kruskal-Wallis 검정

**답 :** ② — 집단마다 크기를 정해 뽑고, 여러 집단의 비율이 같은지 보는 동질성 검정이다.

### 개념 8. 불량 200건 한 표본을 뽑아 '설비'와 '불량 유형' 두 가지로 분류한 뒤, 두 변수가 관련이 있는지 확인하는 검정은?

① 독립성 검정  
② 적합도 검정  
③ 일원분산분석  
④ Tukey HSD

**답 :** ① — 한 표본을 두 범주형 변수로 분류해 관련이 있는지 보는 독립성 검정이다.

### 개념 9. 카이제곱 검정 전에 확인했더니 기대빈도가 5 미만인 칸이 있었다. 알맞은 대처는?

① 그대로 카이제곱 검정을 한다  
② 건수표 대신 비율표를 넣는다  
③ 범주를 합치거나, 2×2 표라면 Fisher 정확검정을 쓴다  
④ 유의수준을 0.10 으로 바꾼다

**답 :** ③ — 기대빈도가 작으면 카이제곱 근사가 부정확해지므로 범주를 합쳐 칸의 건수를 늘리거나 Fisher 정확검정을 쓴다.

### 개념 10. 검정 결과 p = 0.18 이었다 (α = 0.05). 알맞은 결론은?

① 귀무가설이 참임이 증명되었다  
② 귀무가설을 기각하지 못한다. 차이가 있다고 볼 근거가 부족하다  
③ 귀무가설을 기각한다  
④ 대립가설이 참이다

**답 :** ② — 기각하지 못했다고 '같다'가 증명된 것은 아니다. '다르다고 볼 근거가 부족하다'로 쓴다.